In [21]:
import requests
from bs4 import BeautifulSoup
import json
import time
import os
import re

In [22]:
# Where scraped articles are saved — same directory as PetMD so the chunker
# sees all files in one place regardless of which scraper produced them.
OUTPUT_DIR = "data/raw"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Seconds to wait between requests. ASPCA is a non-profit server, so we
# use a slightly longer delay than we did for PetMD out of courtesy.
DELAY = 2.0

# Minimum number of characters a scraped page must contain to be worth saving.
# Pages shorter than this are almost certainly navigation stubs, not real content.
MIN_CONTENT_LENGTH = 200

# Browser-style header so the ASPCA server doesn't reject us as an automated bot.
# ASPCA's server checks this header — without it, requests may return a 403 error.
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# Base URL used to build full links from relative paths found on ASPCA pages.
BASE_URL = "https://www.aspca.org"

In [23]:
def check_robots_txt(base_url: str) -> None:
    """
    Fetch and print the site's robots.txt file before any scraping begins.

    robots.txt is a standard file that tells automated tools which parts of
    a website they are allowed to visit. We print it rather than parsing it
    programmatically so a human can read and confirm compliance before running
    the scraper. ASPCA's pet-care pages are publicly accessible and not blocked.

    Parameters:
        base_url -- The root address of the site, e.g. 'https://www.aspca.org'.
    """
    robots_url = f"{base_url}/robots.txt"

    try:
        response = requests.get(robots_url, headers=HEADERS, timeout=10)
        response.raise_for_status()
        print(f"robots.txt for {base_url}:\n")
        print(response.text)

    except requests.exceptions.RequestException as e:
        # If we can't retrieve robots.txt, we warn and proceed carefully
        # rather than crashing — the scraper still respects polite delays.
        print(f"[WARNING] Could not retrieve robots.txt: {e}")
        print("Proceeding cautiously with polite delays.")


# Always verify compliance before any scraping begins.
check_robots_txt(BASE_URL)

robots.txt for https://www.aspca.org:

#
# robots.txt
#
# This file is to prevent the crawling and indexing of certain parts
# of your site by web crawlers and spiders run by sites like Yahoo!
# and Google. By telling these "robots" where not to go on your site,
# you save bandwidth and server resources.
#
# This file will be ignored unless it is at the root of your host:
# Used:    http://example.com/robots.txt
# Ignored: http://example.com/site/robots.txt
#
# For more information about the robots.txt standard, see:
# http://www.robotstxt.org/robotstxt.html

User-agent: *
Crawl-delay: 10
# CSS, JS, Images
Allow: /misc/*.css$
Allow: /misc/*.css?
Allow: /misc/*.js$
Allow: /misc/*.js?
Allow: /misc/*.gif
Allow: /misc/*.jpg
Allow: /misc/*.jpeg
Allow: /misc/*.png
Allow: /modules/*.css$
Allow: /modules/*.css?
Allow: /modules/*.js$
Allow: /modules/*.js?
Allow: /modules/*.gif
Allow: /modules/*.jpg
Allow: /modules/*.jpeg
Allow: /modules/*.png
Allow: /profiles/*.css$
Allow: /profiles/*.css?
Allo

In [24]:
# ASPCA pages we want to scrape, listed explicitly.
#
# WHY explicit targets instead of crawling?
# ASPCA is not a content farm — it has a small, stable set of high-value
# reference pages. Crawling would pull in donation pages, news posts, and
# other irrelevant content. Explicit targets give us exactly what fills the
# gaps in our PetMD dataset.
#
# WHY the same URL listed twice (once per species)?
# ASPCA writes for dogs and cats together on some pages. Storing each once
# per species lets the chunker and retriever filter by animal cleanly.
#
# ALL URLs below were verified as live on 2026-04-07 via web search.
# The toxic-and-non-toxic-plants page is JS-rendered and returns 0 chars
# with a static scraper — it is intentionally excluded until a Playwright
# solution is available. The /kitten-care and /puppy-care slugs return 404.

ASPCA_TARGETS = [
    # ── Poison Control: People Foods ────────────────────────────────────────
    # Primary data gap: cat/poisoning had zero PetMD articles (404 on PetMD).
    # dog/poisoning also benefits from ASPCA's authoritative toxicity list.
    {
        "url":     "https://www.aspca.org/pet-care/animal-poison-control/people-foods-avoid-feeding-your-pets",
        "title":   "People Foods to Avoid Feeding Your Pets",
        "species": "cat",
        "topic":   "poisoning",
    },
    {
        "url":     "https://www.aspca.org/pet-care/animal-poison-control/people-foods-avoid-feeding-your-pets",
        "title":   "People Foods to Avoid Feeding Your Pets",
        "species": "dog",
        "topic":   "poisoning",
    },

    {
    "url": "https://www.aspca.org/pet-care/aspca-poison-control/poisonous-household-products",
    "title": "Poisonous Household Products",
    "species": "cat",
    "topic": "poisoning",
},
{
    "url": "https://www.aspca.org/pet-care/aspca-poison-control/poisonous-household-products",
    "title": "Poisonous Household Products",
    "species": "dog",
    "topic": "poisoning",
},


    # ── Cat Care ─────────────────────────────────────────────────────────────
    {
        "url":     "https://www.aspca.org/pet-care/cat-care/general-cat-care",
        "title":   "General Cat Care",
        "species": "cat",
        "topic":   "care",
    },
    {
        "url":     "https://www.aspca.org/pet-care/cat-care/cat-nutrition-tips",
        "title":   "Cat Nutrition Tips",
        "species": "cat",
        "topic":   "food_safety",
    },
    {
        "url":     "https://www.aspca.org/pet-care/cat-care/cat-grooming-tips",
        "title":   "Cat Grooming Tips",
        "species": "cat",
        "topic":   "care",
    },
    {
        "url":     "https://www.aspca.org/pet-care/cat-care/common-cat-behavior-issues",
        "title":   "Common Cat Behavior Issues",
        "species": "cat",
        "topic":   "behavior",
    },
    # common-cat-diseases confirmed live via search results (16193 chars already saved).
    {
        "url":     "https://www.aspca.org/pet-care/cat-care/common-cat-diseases",
        "title":   "Common Cat Diseases",
        "species": "cat",
        "topic":   "conditions",
    },

    # ── Dog Care ─────────────────────────────────────────────────────────────
    {
        "url":     "https://www.aspca.org/pet-care/dog-care/general-dog-care",
        "title":   "General Dog Care",
        "species": "dog",
        "topic":   "care",
    },
    {
        "url":     "https://www.aspca.org/pet-care/dog-care/dog-nutrition-tips",
        "title":   "Dog Nutrition Tips",
        "species": "dog",
        "topic":   "food_safety",
    },
    {
        "url":     "https://www.aspca.org/pet-care/dog-care/dog-grooming-tips",
        "title":   "Dog Grooming Tips",
        "species": "dog",
        "topic":   "care",
    },
    {
        "url":     "https://www.aspca.org/pet-care/dog-care/common-dog-behavior-issues",
        "title":   "Common Dog Behavior Issues",
        "species": "dog",
        "topic":   "behavior",
    },
    {
        "url":     "https://www.aspca.org/pet-care/dog-care/common-dog-diseases",
        "title":   "Common Dog Diseases",
        "species": "dog",
        "topic":   "conditions",
    },

    # ── General Pet Care (cross-species) ──────────────────────────────────────
    # Stored as both species because the content applies equally to both.
    {
        "url":     "https://www.aspca.org/pet-care/general-pet-care/behavioral-help-your-pet",
        "title":   "Behavioral Help for Your Pet",
        "species": "cat",
        "topic":   "behavior",
    },
    {
        "url":     "https://www.aspca.org/pet-care/general-pet-care/behavioral-help-your-pet",
        "title":   "Behavioral Help for Your Pet",
        "species": "dog",
        "topic":   "behavior",
    },
]

In [25]:
def _extract_text_from_body(body) -> str:
    """
    Pull readable text out of an HTML content container.

    We try three strategies in order, stopping as soon as we get non-empty text.
    This is necessary because ASPCA pages use different HTML structures
    depending on their content type:
      - Article pages use <p> tags (paragraphs)
      - The toxic plants list uses <li> tags (list items)
      - Some pages mix both

    Returning an empty string (rather than raising an exception) lets the
    caller decide how to handle the failure — keeping this function
    single-purpose and easy to test.

    Parameters:
        body -- A BeautifulSoup HTML element representing the page's content area.

    Returns:
        The extracted text as a single string, or an empty string if nothing
        was found.
    """
    # Strategy 1: Standard paragraphs — works for article-style pages.
    paragraphs = body.find_all("p")
    text = " ".join(p.get_text(strip=True) for p in paragraphs)
    if text.strip():
        return text

    # Strategy 2: List items — works for the toxic plants reference page,
    # which formats each plant entry as a <li> bullet rather than a paragraph.
    list_items = body.find_all("li")
    text = " ".join(item.get_text(strip=True) for item in list_items)
    if text.strip():
        return text

    # Strategy 3: Any text in the container at all.
    # get_text() with a space separator walks every HTML tag and joins their
    # text content — a last resort before we give up.
    text = body.get_text(separator=" ", strip=True)

    # Collapse runs of whitespace left by removed HTML tags into single spaces.
    # e.g. "word   \n  word" → "word word"
    return re.sub(r"\s+", " ", text).strip()


def _find_content_body(soup):
    """
    Locate the HTML element that contains the main article text on an ASPCA page.

    ASPCA's website is built on Drupal CMS. Drupal uses different HTML wrapper
    classes depending on the page template, so we try each known container in
    order from most specific to most general. Returning None (rather than
    raising an error) lets the caller log a useful message and skip the page.

    Parameters:
        soup -- A BeautifulSoup object representing the full parsed HTML page.

    Returns:
        The first matching HTML element, or None if no known container was found.
    """
    # These selectors are ordered from most specific (Drupal field containers)
    # to least specific (the generic <article> tag). We stop at the first match.
    return (
        soup.find("div", class_="field-items")        # Drupal 7 standard body field
        or soup.find("div", class_="field-name-body") # Drupal 7 named field variant
        or soup.find("div", class_="field--name-body")# Drupal 8/9 equivalent
        or soup.find("main")                           # HTML5 semantic main content
        or soup.find("article")                        # HTML5 semantic article tag
    )


def scrape_aspca_page(url: str, title: str, species: str, topic: str) -> dict | None:
    """
    Download and parse a single ASPCA page, returning structured content.

    This is the core function of the scraper. It handles network errors
    gracefully, tries multiple HTML extraction strategies (since ASPCA's
    page layout varies), and returns a dictionary whose shape exactly
    matches the PetMD articles already in data/raw — so the chunker
    downstream doesn't need to treat ASPCA files differently.

    Parameters:
        url     -- Full web address of the ASPCA page to scrape.
        title   -- Human-readable article name (we supply this because
                   ASPCA page <h1> tags can be inconsistent or missing).
        species -- Which animal this content applies to: 'cat' or 'dog'.
        topic   -- Subject category matching the PetMD schema,
                   e.g. 'poisoning', 'care', 'food_safety'.

    Returns:
        A dict with keys: title, content, url, source, species, topic.
        Returns None if the page could not be fetched or yielded no content.
    """
    # ── Step 1: Fetch the page ───────────────────────────────────────────────
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)

        # raise_for_status() converts HTTP error codes (404, 500, etc.) into
        # Python exceptions so we handle them below instead of quietly
        # processing an error page as if it were real content.
        response.raise_for_status()

    except requests.exceptions.Timeout:
        print(f"  [TIMEOUT] Server did not respond within 15 seconds: {url}")
        return None

    except requests.exceptions.HTTPError as e:
        print(f"  [HTTP {e.response.status_code}] {url}")
        return None

    except requests.exceptions.RequestException as e:
        # Catch-all for other network problems: DNS failure, connection reset, etc.
        print(f"  [NETWORK ERROR] {url}: {e}")
        return None

    # ── Step 2: Parse the HTML ───────────────────────────────────────────────
    # BeautifulSoup turns the raw HTML text into a tree we can search.
    # 'html.parser' is Python's built-in parser — no extra dependencies needed.
    soup = BeautifulSoup(response.content, "html.parser")

    # ── Step 3: Find the content container ──────────────────────────────────
    body = _find_content_body(soup)

    if body is None:
        # This means the page structure changed and none of our known selectors
        # matched. Open the URL in a browser, inspect the HTML, and add the
        # correct container class to _find_content_body() above.
        print(f"  [NO BODY] Could not find content container on: {url}")
        print("            Open the page in a browser → right-click the text")
        print("            → Inspect → note the wrapping <div> class, then")
        print("            add it to _find_content_body() and re-run.")
        return None

    # ── Step 4: Extract readable text ───────────────────────────────────────
    content = _extract_text_from_body(body)

    if len(content) < MIN_CONTENT_LENGTH:
        # Content this short is almost certainly a navigation stub or a page
        # that requires JavaScript to fully render its content.
        print(f"  [EMPTY CONTENT] Only {len(content)} chars found on: {url}")
        print("                  The page may be JS-rendered or behind a login.")
        return None

    # ── Step 5: Return structured data ──────────────────────────────────────
    # The shape of this dict intentionally matches the PetMD scraper output
    # so that the chunker treats both sources identically.
    # 'source' is set to 'ASPCA' so citation logic downstream can distinguish
    # it from PetMD articles when constructing answers.
    return {
        "title":   title,
        "content": content,
        "url":     url,
        "source":  "ASPCA",
        "species": species,
        "topic":   topic,
    }

In [26]:
def run_aspca_scraper(targets: list[dict]) -> None:
    """
    Iterate over all ASPCA target pages, scrape each one, and save the
    result as a JSON file in the shared output directory.

    Files that already exist on disk are skipped — this makes the scraper
    safe to re-run after a partial failure without duplicating work or
    overwriting existing files.

    Parameters:
        targets -- A list of dicts, each with keys: url, title, species, topic.
                   See ASPCA_TARGETS for the expected structure.
    """
    total_saved  = 0
    total_skipped = 0
    total_failed  = 0

    for target in targets:
        url     = target["url"]
        title   = target["title"]
        species = target["species"]
        topic   = target["topic"]

        # Build the output filename.
        # The 'aspca_' prefix prevents collisions with PetMD files that may
        # share the same URL slug (e.g. both could have a 'general-cat-care' page).
        # Using species in the name lets us store cat and dog versions of the
        # same page as separate files without overwriting each other.
        slug     = url.rstrip("/").split("/")[-1]
        filename = f"aspca_{species}_{slug}.json"
        filepath = os.path.join(OUTPUT_DIR, filename)

        # Skip files we've already scraped in a previous run.
        # This avoids hammering ASPCA's server unnecessarily.
        if os.path.exists(filepath):
            print(f"  Skipping (already exists): {filename}")
            total_skipped += 1
            continue

        print(f"  Scraping [{species}/{topic}]: {title}")

        article = scrape_aspca_page(
            url=url,
            title=title,
            species=species,
            topic=topic,
        )

        if article is not None:
            # Write the article dict to disk as a human-readable JSON file.
            # ensure_ascii=False preserves accented characters (e.g. plant names
            # like 'Dieffenbachia') without garbling them into escape sequences.
            # indent=2 makes the file easy to read if you open it in a text editor.
            with open(filepath, "w", encoding="utf-8") as out_file:
                json.dump(article, out_file, ensure_ascii=False, indent=2)

            total_saved += 1
            print(f"    Saved ({len(article['content'])} chars): {filename}")

        else:
            total_failed += 1

        # Be a polite visitor — pause between requests so we don't trigger
        # ASPCA's rate-limiting or risk getting our IP address blocked.
        time.sleep(DELAY)

    # Final summary — gives a clear picture of what happened without
    # having to scroll through all the individual log lines.
    print(f"\n{'─' * 40}")
    print(f"Done.")
    print(f"  Saved:   {total_saved}")
    print(f"  Skipped: {total_skipped}  (already existed)")
    print(f"  Failed:  {total_failed}  (see [ERROR] lines above)")
    print(f"{'─' * 40}")


run_aspca_scraper(ASPCA_TARGETS)

  Skipping (already exists): aspca_cat_people-foods-avoid-feeding-your-pets.json
  Skipping (already exists): aspca_dog_people-foods-avoid-feeding-your-pets.json
  Skipping (already exists): aspca_cat_poisonous-household-products.json
  Skipping (already exists): aspca_dog_poisonous-household-products.json
  Skipping (already exists): aspca_cat_general-cat-care.json
  Skipping (already exists): aspca_cat_cat-nutrition-tips.json
  Skipping (already exists): aspca_cat_cat-grooming-tips.json
  Skipping (already exists): aspca_cat_common-cat-behavior-issues.json
  Skipping (already exists): aspca_cat_common-cat-diseases.json
  Skipping (already exists): aspca_dog_general-dog-care.json
  Skipping (already exists): aspca_dog_dog-nutrition-tips.json
  Skipping (already exists): aspca_dog_dog-grooming-tips.json
  Skipping (already exists): aspca_dog_common-dog-behavior-issues.json
  Skipping (already exists): aspca_dog_common-dog-diseases.json
  Skipping (already exists): aspca_cat_behavioral-

In [27]:
# ── QA: Check for empty or very short ASPCA articles ───────────────────────
# We only inspect ASPCA files (those with the 'aspca_' prefix) so this cell
# doesn't re-scan the entire PetMD dataset on every run.

aspca_files = [
    f for f in os.listdir(OUTPUT_DIR)
    if f.startswith("aspca_") and f.endswith(".json")
]

empty_count = 0

for filename in aspca_files:
    with open(os.path.join(OUTPUT_DIR, filename), encoding="utf-8") as fp:
        article = json.load(fp)

    # Articles shorter than 100 characters are almost certainly scraping failures
    # — a real content page will always have more text than this.
    if not article["content"] or len(article["content"]) < 100:
        empty_count += 1
        print(f"  Too short: {article['title']}  ({len(article['content'])} chars)")

print(f"\nTotal ASPCA articles: {len(aspca_files)}")
print(f"Empty or too short:   {empty_count}")


Total ASPCA articles: 16
Empty or too short:   0


In [28]:
# ── QA: Content length distribution ────────────────────────────────────────
# This helps spot pages that scraped successfully but yielded very little text
# — which often means the page is JS-rendered and the static scraper missed
# most of the content.

lengths = []

for filename in aspca_files:
    with open(os.path.join(OUTPUT_DIR, filename), encoding="utf-8") as fp:
        article = json.load(fp)
    lengths.append(len(article["content"]))

if lengths:
    print(f"Shortest: {min(lengths):,} chars")
    print(f"Longest:  {max(lengths):,} chars")
    print(f"Average:  {sum(lengths) // len(lengths):,} chars")
    print(f"Under 500 chars:  {sum(1 for l in lengths if l < 500)}")
    print(f"Under 1000 chars: {sum(1 for l in lengths if l < 1000)}")
else:
    print("No ASPCA articles found. Run the scraper first.")

Shortest: 283 chars
Longest:  23,442 chars
Average:  7,251 chars
Under 500 chars:  1
Under 1000 chars: 2


In [29]:
# ── QA: List all ASPCA articles under 1000 characters ──────────────────────
# Short articles are worth reviewing manually — they may be partially scraped
# pages, stubs, or pages whose layout we haven't handled yet.

SHORT_THRESHOLD = 1000

for filename in sorted(aspca_files):
    with open(os.path.join(OUTPUT_DIR, filename), encoding="utf-8") as fp:
        article = json.load(fp)

    char_count = len(article["content"])
    if char_count < SHORT_THRESHOLD:
        print(
            f"{article['title'][:60]:<60}  "
            f"{char_count:>6} chars  "
            f"{article['species']}/{article['topic']}"
        )

Common Cat Behavior Issues                                       996 chars  cat/behavior
Common Dog Behavior Issues                                       283 chars  dog/behavior


In [30]:
# ── QA: Article distribution by species and topic ──────────────────────────
# Cross-references ASPCA files with the expected targets so we can immediately
# see whether any planned targets failed to save, and whether all required
# metadata fields are present in every file.

from collections import Counter

distribution   = Counter()
missing_fields = []

# The six fields every article — PetMD or ASPCA — must have for the
# chunker and retriever to work correctly.
REQUIRED_FIELDS = ["title", "content", "url", "source", "species", "topic"]

for filename in aspca_files:
    with open(os.path.join(OUTPUT_DIR, filename), encoding="utf-8") as fp:
        article = json.load(fp)

    distribution[(article.get("species", "MISSING"), article.get("topic", "MISSING"))] += 1

    for field in REQUIRED_FIELDS:
        if not article.get(field):
            missing_fields.append((filename, field))

# Print the distribution table.
print(f"{'SPECIES':<14} {'TOPIC':<18} {'COUNT':>6}")
print("-" * 40)
for (species, topic), count in sorted(distribution.items()):
    print(f"{species:<14} {topic:<18} {count:>6}")
print("-" * 40)
print(f"{'TOTAL':<33} {sum(distribution.values()):>6}")

# Report any articles missing required fields.
if missing_fields:
    print(f"\nWARNING: {len(missing_fields)} missing field(s) detected")
    for fname, field in missing_fields[:10]:
        print(f"  {fname} — missing '{field}'")
else:
    print("\nAll articles have required fields. ✓")

# Check for any expected targets that didn't produce a file.
# This catches silent failures where scraping ran but saved nothing.
print("\nCoverage check against ASPCA_TARGETS:")
all_good = True
for target in ASPCA_TARGETS:
    slug     = target["url"].rstrip("/").split("/")[-1]
    filename = f"aspca_{target['species']}_{slug}.json"
    filepath = os.path.join(OUTPUT_DIR, filename)
    status   = "✓" if os.path.exists(filepath) else "✗ MISSING"
    if "MISSING" in status:
        all_good = False
    print(f"  [{status}] {filename}")

if all_good:
    print("\nAll targets saved. ✓")

SPECIES        TOPIC               COUNT
----------------------------------------
cat            behavior                2
cat            care                    2
cat            conditions              1
cat            food_safety             1
cat            poisoning               2
dog            behavior                2
dog            care                    2
dog            conditions              1
dog            food_safety             1
dog            poisoning               2
----------------------------------------
TOTAL                                 16

  aspca_dog_common-dog-behavior-issues.json — missing 'url'

Coverage check against ASPCA_TARGETS:
  [✓] aspca_cat_people-foods-avoid-feeding-your-pets.json
  [✓] aspca_dog_people-foods-avoid-feeding-your-pets.json
  [✓] aspca_cat_poisonous-household-products.json
  [✓] aspca_dog_poisonous-household-products.json
  [✓] aspca_cat_general-cat-care.json
  [✓] aspca_cat_cat-nutrition-tips.json
  [✓] aspca_cat_cat-grooming-tip